# Read lagrangian trajectories and compute statistics from relative dispersion

## Read data and get all pairs of particles

The data (trajectories) consist in netcdf files available separately on Zenodo given their large size:
- expA.np4000.nt6000.nc 
- expAnomean.np4000.nt6000.nc 
- expB.np4000.nt6000.nc 
- expBnomean.np4000.nt6000.nc
- expN.np4000.nt6000.nc 
- expNnomean.np4000.nt6000.nc
- expO.np4000.nt6000.nc 
- expOnomean.np4000.nt6000.nc

"np" is the number of particles released, "nt" is the number of time steps available. "nomean" means that the mean flow has be subtracted from the PIV velocity fields before advecting the particles.

In [ ]:
import numpy as np
import xarray as xr
import sys
sys.path.append('../')
from xdispersion import RelativeDispersion
from matplotlib import cm
import matplotlib.pyplot as plt
import pandas as pd

save=False  #save figures or not
namedata = "ExpA" #ExpA or ExpAnomean, ExpB or ExpBnomean, etc
drifters = xr.open_dataset('../data_RSTA25_Lemasquerier/data/ExpA/fort.3001.np4000nt6000.nc')


Now we initialize a `RelativeDispersion` class, with the `drifters` dataset and associated names for position and velocity. 

In [ ]:

rd = RelativeDispersion(drifters, ragged=False, ID='tracer',
                        xpos='x', uvel='ux', time='time',
                        ypos='y', vvel='uy', coord='polar')

print(rd)

pairs = rd.get_all_pairs()
pairs


## Plot a histogram of initial separation distances

In [ ]:

r0_min = pairs.r0.min().item()
r0_max = pairs.r0.max().item()

fig, ax = plt.subplots(figsize=(6.5, 2.5), facecolor='w')

ax.hist(pairs.r0, bins=np.linspace(0, r0_max, 51), rwidth=0.86)
ax.set_xlabel('r0 (m)')
ax.set_title('histogram of initial separations')
plt.tight_layout()
plt.show()

## Select pairs based on initial separation

In [ ]:
origin_pairs = rd.get_original_pairs(pairs, r0=[0.005, 0.01])
# This is equivalent to:
# condition = np.logical_and(pairs.r0>=0.005, pairs.r0<=0.01)
# origin_pairs = pairs.where(condition, drop=True).astype(pairs.dtypes)

origin_pairs

## Calculate time-based and separation-based measures

In [ ]:
from xdispersion import gen_rbins, rel_disp, rel_diff, kurtosis, famp_growth_rate

# separation bins (geometric factor alpha)
alpha = 1.2
rbins = gen_rbins(0.01, 0.5, alpha)

# Get all building blocks for the time-based measures. 
# Here, rx is radial dispersion, ry is zonal dispersion and r is total dispersion
rx, ry, rxy, r, rpb = rd.separation_measures(origin_pairs)

r2   = rel_disp(r, order=2, mean_at='const-t') #order=2 so 2nd moment of separation
K2   = rel_diff(r, mean_at='const-t') #Relative diffusivity
Ku   = kurtosis(r, mean_at='const-t') #kurtosis=4th moment

#mean square separation
rtmp = np.sqrt(r2)

#Calculate finite amplitude growth rate (FAGR)
FAGR = famp_growth_rate(r, mean_at='const-r', rbins=rbins)


We can also calculate the pFSLE from relative diffusivity:

In [ ]:
pFSLE=K2.values/rtmp.values**2 #this has indeed dimension of inverse time

# Digitize rtmp values into bins
bin_indices = np.digitize(rtmp.values, rbins)
# Prepare array to store binned mean
pFSLE_binned_mean = np.full(len(rbins[1:]), np.nan)

# Compute mean pFSLE in each bin
for i in range(1, len(rbins)):
    mask = bin_indices == i
    if np.any(mask):
        pFSLE_binned_mean[i - 1] = np.nanmean(pFSLE[mask])


### Physical fitted values for energy dissipation rate (epsilon), diffusivity (kappa), non-local time T...

In [ ]:
# Get idx corresponding to experiment
mapping = {
    'ExpA': 0, 'ExpB': 1, 'ExpN': 2, 'ExpO': 3,
    'ExpAnomean': 4, 'ExpBnomean': 5, 'ExpNnomean': 6, 'ExpOnomean': 7}

idx = mapping.get(namedata, -1)  # default to -1 if not found

    
    
#Values determined from fits: ExpA/B/N/O. 
# beta=epsilon^(1/3) with epsilon energy dissipation rate
betaall=[9e-3,8e-3,2.5e-3,8e-4,4.5e-3,4.5e-3,1.5e-3,4.3e-4]
betaminall=[7e-3,6e-3,1.5e-3,6e-4,3e-3,3e-3,1e-3,3e-4]
betamaxall=[12e-3,11e-3,4e-3,1e-3,6e-3,6e-3,2e-3,6e-4]

#kappa is diffusivity
kappalall=[5e-4,5e-4,1e-4,2e-5,1.8e-4,1.6e-4,3.5e-5,6e-6]
kappalminall=[3e-4,3e-4,7e-5,1.3e-5,1e-4,1e-4,1.5e-5,4e-6]
kappalmaxall=[8e-4,7e-4,1.5e-4,3.3e-5,3e-4,2.5e-4,6e-5,9e-6]

# T=eta^(-1/3) where eta is the enstrophy dissipation rate
Tall=[9.84,10.8,22.7,105,11.8,13.0,45.7,152,14]
Tminall=[9.74,10.4,22.5,105,11.4,12.7,45.5,150]
Tmaxall=[9.95,11.0,22.9,106,12.0,13.2,46.0,153]

beta=betaall[idx] 
betamin=betaminall[idx]
betamax=betamaxall[idx]

kappal=kappalall[idx] 
kappalmin=kappalminall[idx]
kappalmax=kappalmaxall[idx]

T=Tall[idx] #seconds





## Plot dispersion, relative diffusivity and kurtosis

In [ ]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

fig, axes = plt.subplots(1, 3, figsize=(12, 5))  # (rows, cols)

# First subplot (dispersion)
ax = axes[0]  
ax.scatter(r2.rtime,r2.values,marker='x',s=5,color='b')
ax.fill_between(r2.rtime[40:], 5.2675*(betamin)**3*r2.rtime[40:]**3,
                 5.2675*(betamax)**3*r2.rtime[40:]**3, color='r', alpha=0.15,
                  edgecolor=None, label=None)
ax.plot(r2.rtime[40:],5.2675*(beta)**3*r2.rtime[40:]**3,color='r',
        label=fr'$5.2675 \epsilon t^3$ ($\epsilon$={beta**3:.1e} m$^{{2}}$s$^{{-3}}$)',
        linewidth=1.0)
ax.legend(loc='upper left')
ax.set_ylim(1e-5,max(r2.values)*100)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r"$t$ (s)")
ax.set_ylabel(r"$\langle r^2 \rangle$ (m$^2$)")
ax.set_title("Relative Dispersion")

# Second subplot (relative diffusivity)
axes[1].scatter(rtmp.values,K2.values,marker='x',s=5,color='b')
# Plot shaded region
axes[1].fill_between(rtmp, 2.61*betamin*rtmp**(4/3), 2.61*betamax*rtmp**(4/3),
                      color='r', alpha=0.15, edgecolor=None, label=None)
axes[1].fill_between(rtmp, rtmp*0+2*kappalmin, rtmp*0+2*kappalmax,
                      color='k', alpha=0.1, edgecolor=None, label=None)
axes[1].plot(rtmp,2.61*beta*rtmp**(4/3),color='r',
             label=r'$2.61 \epsilon^{1/3} r^{4/3}$',linewidth=1.0)
axes[1].plot(rtmp,rtmp*0+2*kappal,'k',
             label=fr'$2\kappa$ ($\kappa$ = {kappal:.1e} m$^2$s$^{{-1}}$)',linewidth=1.0)
axes[1].set_title(r"Relative Diffusivity")
axes[1].set_ylim(ymin=1e-8)
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel(r"$r$ (m)")
axes[1].set_ylabel(r"$K_2$ (m$^2$s$^{{-1}}$)")
axes[1].legend()

# Third subplot (Kurtosis)
axes[2].scatter(Ku.rtime, Ku.values, marker='x', s=5, color='b')
axes[2].plot(Ku.rtime,Ku.rtime*0+2,'--k')
axes[2].plot(Ku.rtime,Ku.rtime*0+5.6,'-.k')
axes[2].plot(Ku.rtime.values[:500], np.exp(8 * Ku.rtime.values[:500]/T), 'r-', label=fr'$\exp(8t/T)$ (T = {T:.2f} s)')
axes[2].set_title("Kurtosis")
axes[2].set_xlabel(r"$t$ (s)")
axes[2].set_ylabel(r"$Ku$")
axes[2].set_ylim(0,max(Ku.values)*1.1)
axes[2].legend(loc='lower right')


# --- ADD INSET ON THIRD SUBPLOT ---
axins = inset_axes(axes[2], width="45%", height="35%", loc='upper right',
                   borderpad=1.2)  # width and height as percentages of parent
mask = Ku.rtime <= T/2 # Plot the same data, but limit to t <= T/2
axins.scatter(Ku.rtime[mask], Ku.values[mask], marker='x', s=5, color='b')
axins.plot(Ku.rtime[mask], np.exp(8 * Ku.rtime[mask] / T), 'r-', linewidth=1.0)
axins.set_xlim(0, T/2)
axins.set_ylim(min(Ku.values[mask])*0.9, max(Ku.values[mask])*1.1)

# --- Add subplot labels (a), (b), (c) ---
fig.text(0.02, 0.86, "(a)", fontsize=12)
fig.text(0.34, 0.86, "(b)", fontsize=12)
fig.text(0.68, 0.86, "(c)", fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.93])  # leave space at top for labels
if save:
    output_path = f"./output_figures/basic-statistics_{namedata}.pdf"
    fig.savefig(output_path, format='pdf', bbox_inches='tight')  # save as PDF
plt.show()






## Compute the CDF and plot it, fit an exponential decay to measure the cumulative inverse separation time (CIST)

In [ ]:
from xdispersion import * 

PDF=prob_dens_func(r,rbins) #probability density function
CDF = cumul_dens_func(PDF,rbins) #cumulative density function



### Define exponential fit

In [ ]:

from typing import Tuple
from scipy.optimize import curve_fit
import numpy as np

def exp_decay_offset_fit(t: np.array, y: np.array) -> Tuple[float, float, float, float, float]:
    """Fit y = A * exp(-(t - t0) / tau) + C """
    idx = ~np.isnan(y)
    t_fit = t[idx]
    y_fit = y[idx]

    if len(y_fit) < 4:
        return np.nan, np.nan, np.nan, np.nan, np.nan

    try:
        def model(t, A, tau, t0, C):
            t_shifted = t - t0
            return np.where(t_shifted >= 0, A * np.exp(-t_shifted / tau) + C, A + C)

        # Initial guesses: [A, tau, t0, C]
        A0 = y_fit.max() - y_fit.min()
        tau0 = (t_fit.max() - t_fit.min()) / 3
        t0_0 = t_fit.min()
        C0 = y_fit.min()
       
        # Bounds: A free, tau>0, t0>=0, C free
        lower_bounds = [-np.inf, 1e-12, 0, -np.inf]
        upper_bounds = [ np.inf,  np.inf, np.inf,  np.inf]
        
        popt, _ = curve_fit(model, t_fit, y_fit, p0=[A0, tau0, t0_0, C0],bounds=(lower_bounds, upper_bounds), maxfev=10000)
        A, tau, t0, C = popt


        y_pred = model(t_fit, A, tau, t0, C)
        rmse = np.sqrt(np.mean((y_pred - y_fit) ** 2))
        return A, tau, t0, C, rmse

    except:
        return np.nan, np.nan, np.nan, np.nan, np.nan



### Apply the exponential fit 

In [ ]:


# Exclude the first rtime point for fitting
CDF_fit = CDF.isel(rtime=slice(1, None))
rtime_fit = CDF_fit['rtime']


# Apply the fitting function on total dispersion
A_exp, tau_exp, t0_exp, C_exp, rmse_exp = xr.apply_ufunc(
    exp_decay_offset_fit,
    rtime_fit,
    CDF_fit,
    dask='allowed',
    input_core_dims=[['rtime'], ['rtime']],
    output_core_dims=[[], [], [], [], []],
    vectorize=True
)




### Plot CDF with exponential fit and save figure

In [ ]:

# Select fixed rbin values for the plot
rbins_to_plot = [rbins.values[8],rbins.values[11], rbins.values[15], rbins.values[17], rbins.values[19]]  # example rbin values
colors = cm.viridis(np.linspace(0, 1, len(rbins_to_plot)))

fig=plt.figure(figsize=(7, 5))
for i, rb in enumerate(rbins_to_plot):
    # Raw CDF
    cdf_line = CDF_fit.sel(rbin=rb, method='nearest')
    rtime_vals = cdf_line['rtime'].values
    cdf_vals = cdf_line.values
    rb_val = cdf_line.rbin.values

    plt.plot(rtime_vals, cdf_vals, 'x', markersize=3, color=colors[i], label=f'rbin={rb_val:.2f} m')

    # Exponential fit
    A = A_exp.sel(rbin=rb, method='nearest').values
    tau = tau_exp.sel(rbin=rb, method='nearest').values
    t0 = t0_exp.sel(rbin=rb, method='nearest').values
    C = C_exp.sel(rbin=rb, method='nearest').values

    if np.all(np.isfinite([A, tau, t0, C])):
        t_vals = rtime_vals
        t_shifted = t_vals - t0
        exp_fit = np.where(t_shifted >= 0, A * np.exp(-t_shifted / tau) + C, A + C)
    
        label_exp = f'Exponential fits' if i == len(rbins_to_plot)-1 else None
        plt.plot(t_vals, exp_fit, '--', color='black', label=label_exp)

plt.plot(CDF.rtime,CDF.rtime*0+0.5,color='black',linestyle=':')
plt.xlabel('t (s)')
plt.ylabel('CDF')
plt.title('CDF with Exponential Fits')
plt.legend()
plt.grid(True)
plt.tight_layout()

if save:
    output_path = f"./output_figures/CDF_{namedata}.pdf"
    fig.savefig(output_path, format='pdf', bbox_inches='tight')  # save as PDF
plt.show()

### Compute half time from exponential fit

In [ ]:
# Compute t₀.₅ (time where CDF = 0.5)
valid_mask = np.logical_and(
    A_exp > 0,
    np.logical_and(tau_exp > 0, (0.5 - C_exp) / A_exp > 0)
)

t_half_exp = xr.where(
    valid_mask,
    t0_exp - tau_exp * np.log((0.5 - C_exp) / A_exp),
    np.nan
)

# Compute Δt_half and its inverse
dt_half = t_half_exp.diff('rbin')
CIST_exp = 1.0 / dt_half

# rbin values corresponding to upper bins in diff
rbin_upper = t_half_exp['rbin'][1:]

In [ ]:
# Find the index of the value in the CDF nearest to 0.5 (to compare with the fit)
idx_nearest = abs(CDF - 0.5).argmin(dim='rtime')
t_half_actual = CDF['rtime'].isel(rtime=idx_nearest)

# Find rbin_max where t_half_actual is closest to tmax 
tmax = CDF.rtime.values.max()
tol = 0.1 * tmax # Define tolerance
mask = np.abs(t_half_actual - tmax) <= tol # Mask rbin values within ±5% of tmax
rbin_candidates = CDF.rbin.where(mask, drop=True) # Get the smallest rbin that satisfies the condition
if rbin_candidates.size > 0:
    rbin_max = rbin_candidates.min().item()
else:
    rbin_max = CDF.rbin.max().item()

## Final CIST plot with FAGR and pFSLE

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 5))

ax.grid(True, which='both', linewidth=0.5, color='lightgrey')
ax.plot(rbin_upper[rbin_upper <= rbin_max], CIST_exp[rbin_upper <= rbin_max],
         'xk', markeredgewidth=1.5, label='CIST', zorder=4)
ax.plot(rbins[1:], pFSLE_binned_mean, 'o', color='k', markersize=4, label='pFSLE', zorder=3)
ax.plot(FAGR.rbin, FAGR.values, 'd', markersize=3, color=(0.6, 0.6, 0.6), label='FAGR')

# Theoretical prediction for diffusion
ax.plot(rbins[2:24], 4*kappal*np.log(2)/(alpha**2-1)*rbins[2:24]**(-2), 'r-', 
        label=r'$4\kappa \frac{\ln(2)}{ \alpha^2 - 1}  r^{-2}$', linewidth=1.0)
ax.fill_between(rbins[2:], 4*kappalmin*np.log(2)/(alpha**2-1)*rbins[2:]**(-2), 
                4*kappalmax*np.log(2)/(alpha**2-1)*rbins[2:]**(-2), color='r', 
                alpha=0.1, edgecolor=None, label=None)

# Theoretical prediction for Richardson
ax.plot(rbin_upper[0:17], 4*beta*2.6741/(9*(alpha**(2/3)-1))*rbin_upper[0:17]**(-2/3),
         'y-', label=r'$4\epsilon^{1/3}  \frac{2.6741}{ 9 (\alpha^{2/3} - 1)}  r^{-2/3}$',
         linewidth=1.0)
ax.fill_between(rbin_upper[0:17], 4*betamin*2.6741/(9*(alpha**(2/3)-1))*rbin_upper[0:17]**(-2/3), 
                4*betamax*2.6741/(9*(alpha**(2/3)-1))*rbin_upper[0:17]**(-2/3), color='y', 
                alpha=0.15, edgecolor=None, label=None)

# Theoretical prediction for enstrophy range
ax.plot(rbin_upper[:5], rbin_upper[:5]*0 + 2/T/np.log(alpha), 'k-',
        label=r'$2 / (T \ln(\alpha))$', linewidth=1.0)

# Shade region where rbin > rbin_max
ax.axvspan(rbin_max, 1.1 * rbins.max(), color='gray', alpha=0.3) 

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('rbin (m)')
ax.set_ylabel(r'CIST and FLSE (s$^{-1}$)')
ax.set_ylim(10**(-3), 2)
ax.set_xlim(rbins.min(), 1.1 * rbins.max())
ax.set_title(f"{namedata} total dispersion")
ax.legend(loc='upper right')

plt.tight_layout()
if save:
    output_path = f"./output_figures/CISTandFAGR_err_{namedata}.pdf"
    fig.savefig(output_path, format='pdf', bbox_inches='tight')
plt.show()
